# Phenotypes: plausible range, residualise, trim, standardise within sex

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = cfg.params.get("threads", 4)

def setenv(**kw):
    """Variables for the %%bash / %%script cells below."""
    os.environ.update({k: str(v) for k, v in kw.items()})

def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell or R cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

## Functions

In [ ]:
def hist_safe(ax, x, bins=60, **kw):
    """Histogram with bins of 1-20 participants left blank."""
    x = np.asarray(x, dtype=float)
    lo, hi = np.quantile(x[np.isfinite(x)], [0.005, 0.995])
    counts, edges = np.histogram(x, bins=np.linspace(lo, hi, bins + 1))
    ax.stairs(np.where(counts > 20, counts, 0), edges, fill=True, alpha=0.6, **kw)

In [ ]:
STAGE = "phenotypes"
require_ready(cfg, STAGE)
rep = Report(cfg, STAGE)
items = target_items(cfg, STAGE)
OUT_DIR = os.path.dirname(items[0][0])
RUN = FORCE or bool(stage_status(cfg, STAGE).missing)
setenv(RUN=int(RUN), LIB=os.path.join(REPO, "scripts/local/residualize_lib.R"),
       TABLE_DIR=cfg.input("phenotype_tables"), PHENO_LIST=cfg.input("phenotype_list"),
       PHENOS=",".join(cfg.lists["phenotypes"]), TRANSFORMS=",".join(cfg.lists["transforms"]),
       COVSETS=",".join(cfg.lists["covsets"]), KEEP=single_target(cfg, "ancestry_round2"),
       PCS=single_target(cfg, "covariate_pcs"), N_PCS=cfg.param("cov_n_pcs"),
       OUTLIER_SD=cfg.param("outlier_sd"), OUT_DIR=OUT_DIR)

## Residualise (residualize_lib.R, shared with the CEU/GBR pipeline)

In [ ]:
%%script Rscript -
suppressMessages({ library(data.table); library(dplyr) })
if (Sys.getenv("RUN") != "1") { cat("exists, keeping it (FORCE = True to remake)\n"); quit(save = "no") }
env <- function(k) { v <- Sys.getenv(k); if (!nzchar(v)) stop("missing ", k); v }
split <- function(k) strsplit(env(k), ",")[[1]]
source(env("LIB"))
out_dir <- env("OUT_DIR"); dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

keep <- fread(env("KEEP"), header = FALSE, colClasses = "character")$V1
pcs <- fread(env("PCS"), colClasses = list(character = "person_id"))
covsets <- list(base = "age", base_pcs = c("age", paste0("PC", seq_len(as.integer(env("N_PCS"))))))
plist <- fread(env("PHENO_LIST"), sep = "\t")

qc <- list()
for (ph in split("PHENOS")) {
  row <- plist[phenotype_name == ph]
  tab <- fread(file.path(env("TABLE_DIR"), paste0(ph, ".tsv")), colClasses = list(character = "person_id"))
  n_table <- nrow(tab)
  tab <- tab[person_id %in% keep]
  pr <- filter_plausible_range(as.data.frame(tab), "phenotype", row$plausible_min, row$plausible_max)
  df <- add_transformed_variant(as.data.frame(merge(as.data.table(pr$data), pcs, by = "person_id")), "phenotype")
  for (tr in split("TRANSFORMS")) {
    col <- if (tr == "raw") "phenotype" else "phenotype__invnorm"
    for (cs in split("COVSETS")) {
      res <- residualize_phenotype(df, col, "sex_at_birth", covsets[[cs]], as.numeric(env("OUTLIER_SD")))
      # FID = 0 to match plink's .grm.id
      out <- res$data %>% filter(!is.na(phenotype_norm)) %>% transmute(FID = 0, IID = person_id, Y = phenotype_norm)
      write.table(out, file.path(out_dir, sprintf("%s__%s__%s.pheno", ph, tr, cs)),
                  quote = FALSE, sep = " ", row.names = FALSE)
      qc[[length(qc) + 1]] <- data.table(phenotype = ph, transform = tr, covset = cs, n_table = n_table,
        n_in_set = nrow(tab), n_implausible = pr$n_excluded, n_with_pcs = nrow(df), n_retained = res$n_retained,
        n_distinct = length(unique(df[[col]])), r_squared = res$r_squared,
        skew_input = e1071::skewness(df[[col]], na.rm = TRUE))
    }
  }
}
fwrite(rbindlist(qc), file.path(out_dir, "qc_summary.tsv"), sep = "\t")

In [ ]:
if RUN:
    for uri, item in items:
        write_meta(get_store(uri), uri, collect_meta(cfg, STAGE, uri, item=item))

## Checks

In [ ]:
qc = pd.read_csv(f"{OUT_DIR}/qc_summary.tsv", sep="\t")
MIN_N = cfg.params.get("pheno_min_n", 1000)
for r in qc.itertuples():
    rep.check(f"{r.phenotype}/{r.transform}/{r.covset}: N retained >= {MIN_N}", r.n_retained >= MIN_N)
for ph, d in qc.groupby("phenotype"):
    r = d.iloc[0]
    rep.check(f"{ph}: not near-binary (>= 50 distinct values)", r.n_distinct >= 50)
    rep.check(f"{ph}: implausible values < 1%", r.n_implausible / r.n_in_set < 0.01,
              f"{r.n_implausible / r.n_in_set:.2%}")
for uri, item in items:
    y = pd.read_csv(uri, sep=" ")["Y"]
    rep.check(f"{os.path.basename(uri)}: mean 0, SD 1", abs(y.mean()) < 0.05 and abs(y.std() - 1) < 0.05,
              f"{y.mean():.3f}, {y.std():.3f}")
rep.table(qc.round(3), "qc_summary", counts=["n_table", "n_in_set", "n_implausible", "n_with_pcs", "n_retained"])

In [ ]:
fig, axes = plt.subplots(1, len(cfg.lists["phenotypes"]), figsize=(12, 4))
for ax, ph in zip(np.atleast_1d(axes), cfg.lists["phenotypes"]):
    for tr in cfg.lists["transforms"]:
        hist_safe(ax, pd.read_csv(f"{OUT_DIR}/{ph}__{tr}__base_pcs.pheno", sep=" ")["Y"], label=tr)
    ax.set(title=ph, xlabel="residualised, standardised"); ax.legend()
rep.fig(fig, "cleaned_distributions"); plt.show()
rep.save()